# Time series 5: ARIMA only

Fit **ARIMA(p,d,q)** per frequency band on train data, then **rolling 1-step-ahead** forecast on the test set (last 8760 hours). Same evaluation: MAE and MASE vs naive.

Note: ARIMA is fit once per band; then we append each test observation and forecast the next step (no re-estimation). Can be slow with many bands.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

_root = Path.cwd().resolve()
if _root.name == "time_series":
    _root = _root.parent
DATA_PATH = _root / "data" / "transformed" / "transformed_data.parquet"
if not DATA_PATH.exists():
    DATA_PATH = DATA_PATH.with_suffix(".csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Transform data not found at {DATA_PATH}. Run task transform first.")

df = pd.read_parquet(DATA_PATH) if DATA_PATH.suffix == ".parquet" else pd.read_csv(DATA_PATH)
df = df.sort_values(["date", "hour"]).reset_index(drop=True)
df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")

freq_bands = sorted([c for c in df.columns if c not in ("date", "hour", "datetime")])
TEST_STEPS = 24 * 365

print(f"Loaded {len(df)} rows. Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Frequency bands: {len(freq_bands)}")
print(f"Test window: last {TEST_STEPS} hours (1-step-ahead)")

Loaded 9192 rows. Date range: 2025-01-01 to 2026-01-18
Frequency bands: 70
Test window: last 8760 hours (1-step-ahead)


## Fit ARIMA per band, rolling 1-step forecast on test

ARIMA(p,d,q) via SARIMAX with no seasonal part: order=(p,d,q), seasonal_order=(0,0,0,0). We use (1,0,1) for speed; you can try (2,0,2) or (1,1,1) if desired.

In [6]:
ORDER = (1, 0, 1)  # (p, d, q) for ARIMA
rows_arima = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 50:
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    train_vals = vals[:-TEST_STEPS]
    test_vals = vals[-TEST_STEPS:]
    test_dts = dts[-TEST_STEPS:]
    try:
        model = SARIMAX(train_vals, order=ORDER, seasonal_order=(0, 0, 0, 0), enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit(disp=False)
    except Exception as e:
        continue
    for i in range(TEST_STEPS):
        try:
            pred = res.forecast(steps=1)[0]
            if i >= 1:
                rows_arima.append({"datetime": test_dts[i], "frequency_band": freq, "actual": float(test_vals[i]), "predicted": float(pred)})
            res = res.append([test_vals[i]], refit=False)
        except Exception:
            break
    if (freq_bands.index(freq) + 1) % 20 == 0:
        print(f"  ARIMA done for {freq_bands.index(freq) + 1}/{len(freq_bands)} bands")

df_arima = pd.DataFrame(rows_arima)
print(f"ARIMA: {len(df_arima)} predictions")

  ARIMA done for 20/70 bands


/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  ARIMA done for 40/70 bands
  ARIMA done for 60/70 bands
ARIMA: 613130 predictions


## Naive baseline and MASE

In [7]:
rows_naive = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 1:
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    test_vals, test_dts = vals[-TEST_STEPS:], dts[-TEST_STEPS:]
    for i in range(0, TEST_STEPS - 1):
        rows_naive.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i]})
df_naive = pd.DataFrame(rows_naive)

def mase_from_df(pred_df):
    diffs = pred_df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(pred_df["actual"] - pred_df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

mae_naive = float(np.mean(np.abs(df_naive["actual"] - df_naive["predicted"])))
mase_naive = mase_from_df(df_naive)
print(f"Naive:  MAE = {mae_naive:.4f},  MASE = {mase_naive:.4f}" if not np.isnan(mase_naive) else f"Naive:  MAE = {mae_naive:.4f}")

Naive:  MAE = 2.4198,  MASE = 0.9999


## ARIMA vs Naive

In [8]:
if len(df_arima) == 0:
    print("No ARIMA predictions (fit failed for all bands). Try smaller order or check data.")
else:
    mae_arima = float(np.mean(np.abs(df_arima["actual"] - df_arima["predicted"])))
    mase_arima = mase_from_df(df_arima)
    print("=== 1-step-ahead: ARIMA vs Naive ===")
    print(f"  Naive:  MAE = {mae_naive:.4f},  MASE = {mase_naive:.4f}" if not np.isnan(mase_naive) else f"  Naive:  MAE = {mae_naive:.4f}")
    print(f"  ARIMA:  MAE = {mae_arima:.4f},  MASE = {mase_arima:.4f}" if not np.isnan(mase_arima) else f"  ARIMA:  MAE = {mae_arima:.4f}")
    if mae_arima < mae_naive:
        print("  → ARIMA beats naive.")
    else:
        print("  → Naive best (common at 1-step for persistent series).")

=== 1-step-ahead: ARIMA vs Naive ===
  Naive:  MAE = 2.4198,  MASE = 0.9999
  ARIMA:  MAE = 2.4779,  MASE = 1.0239
  → Naive best (common at 1-step for persistent series).


## Blend ARIMA with winning blend (0.91×naive + 0.09×seasonal)

Blend ARIMA with the winning blend (α=0.91) and tune the mix to see if we can beat naive. All MAE/MASE below are computed on the **12-month test window** (last 8760 hours). Uses existing `df_arima` so no re-fit.

In [10]:
# 12-month test window = last TEST_STEPS hours (8760 = 365*24)
if len(df_arima) == 0:
    print("No ARIMA predictions; run the ARIMA cell first.")
else:
    ALPHA_BLEND = 0.91
    rows_naive_b, rows_seasonal_b = [], []
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        if len(sub) < TEST_STEPS + 24:
            continue
        vals = sub[freq].values.astype(np.float64)
        dts = sub["datetime"].values
        test_vals, test_dts = vals[-TEST_STEPS:], dts[-TEST_STEPS:]
        for i in range(0, TEST_STEPS - 1):
            rows_naive_b.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i]})
        for i in range(23, TEST_STEPS - 1):
            rows_seasonal_b.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i-23]})
    df_naive_b = pd.DataFrame(rows_naive_b).rename(columns={"predicted": "p_naive"})[["datetime", "frequency_band", "actual", "p_naive"]]
    df_seasonal_b = pd.DataFrame(rows_seasonal_b).rename(columns={"predicted": "p_seasonal"})[["datetime", "frequency_band", "p_seasonal"]]
    merge_b = df_naive_b.merge(df_seasonal_b, on=["datetime", "frequency_band"], how="inner")
    merge_b["blend_pred"] = ALPHA_BLEND * merge_b["p_naive"] + (1 - ALPHA_BLEND) * merge_b["p_seasonal"]

    # Same rows as df_arima (12-month test window)
    combo = merge_b.merge(
        df_arima.rename(columns={"predicted": "pred_arima"})[["datetime", "frequency_band", "pred_arima"]],
        on=["datetime", "frequency_band"], how="inner"
    )
    n_12m = len(combo)
    # MAE / MASE on this 12-month subset
    mae_naive_12 = float(np.mean(np.abs(combo["actual"] - combo["p_naive"])))
    mae_arima_12 = float(np.mean(np.abs(combo["actual"] - combo["pred_arima"])))
    mae_blend_12 = float(np.mean(np.abs(combo["actual"] - combo["blend_pred"])))
    def _mase(sub_df):
        d = sub_df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
        denom = float(d.mean()) if len(d) > 0 else np.nan
        mae = float(np.mean(np.abs(sub_df["actual"] - sub_df["predicted"])))
        return mae / denom if denom and denom > 0 else np.nan
    mase_naive_12 = _mase(combo.assign(predicted=combo["p_naive"])[["datetime", "frequency_band", "actual", "predicted"]])
    mase_arima_12 = _mase(combo.assign(predicted=combo["pred_arima"])[["datetime", "frequency_band", "actual", "predicted"]])
    mase_blend_12 = _mase(combo.assign(predicted=combo["blend_pred"])[["datetime", "frequency_band", "actual", "predicted"]])

    # Tune γ: combo_pred = γ*blend + (1-γ)*arima
    best_gamma, best_mae = 0.5, np.inf
    for gamma in np.linspace(0, 1, 21):
        pred_c = gamma * combo["blend_pred"] + (1 - gamma) * combo["pred_arima"]
        mae_c = float(np.mean(np.abs(combo["actual"] - pred_c)))
        if mae_c < best_mae:
            best_mae, best_gamma = mae_c, gamma
    combo["pred_combo"] = best_gamma * combo["blend_pred"] + (1 - best_gamma) * combo["pred_arima"]
    mase_combo_12 = _mase(combo.rename(columns={"pred_combo": "predicted"})[["datetime", "frequency_band", "actual", "predicted"]])

    print("=== 12-month test window: Naive, ARIMA, Blend (α=0.91), Blend+ARIMA ===")
    print(f"  Rows (bands × hours): {n_12m}")
    print(f"  Naive:         MAE = {mae_naive_12:.4f},  MASE = {mase_naive_12:.4f}" if not np.isnan(mase_naive_12) else f"  Naive:         MAE = {mae_naive_12:.4f}")
    print(f"  ARIMA:         MAE = {mae_arima_12:.4f},  MASE = {mase_arima_12:.4f}" if not np.isnan(mase_arima_12) else f"  ARIMA:         MAE = {mae_arima_12:.4f}")
    print(f"  Blend (0.91):  MAE = {mae_blend_12:.4f},  MASE = {mase_blend_12:.4f}" if not np.isnan(mase_blend_12) else f"  Blend (0.91):  MAE = {mae_blend_12:.4f}")
    print(f"  Blend+ARIMA (γ={best_gamma:.2f}): MAE = {best_mae:.4f},  MASE = {mase_combo_12:.4f}" if not np.isnan(mase_combo_12) else f"  Blend+ARIMA (γ={best_gamma:.2f}): MAE = {best_mae:.4f}")
    best_mae_12 = min(mae_naive_12, mae_arima_12, mae_blend_12, best_mae)
    if best_gamma >= 0.99:
        print("  → Best γ=1: blend only; adding ARIMA did not help (blend already beats naive).")
    elif best_gamma <= 0.01:
        print("  → Best γ=0: ARIMA only; blend did not help.")
    elif best_mae < mae_naive_12:
        print("  → Blend+ARIMA (actual mix) beats naive on this 12-month window.")
    elif mae_blend_12 == best_mae_12:
        print("  → Blend (0.91) best.")
    elif mae_naive_12 == best_mae_12:
        print("  → Naive best (common at 1-step for persistent series).")
    else:
        print("  → ARIMA best on this subset.")

=== 12-month test window: Naive, ARIMA, Blend (α=0.91), Blend+ARIMA ===
  Rows (bands × hours): 611520
  Naive:         MAE = 2.4178,  MASE = 0.9999
  ARIMA:         MAE = 2.4758,  MASE = 1.0239
  Blend (0.91):  MAE = 2.3900,  MASE = 0.9884
  Blend+ARIMA (γ=1.00): MAE = 2.3900,  MASE = 0.9884
  → Best γ=1: blend only; adding ARIMA did not help (blend already beats naive).
